In [1]:
## XGBoost Classifier Implementation
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings(action='ignore')

%matplotlib inline

In [2]:
df = pd.read_csv('Travel.csv')
df.head()

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [3]:
df['Gender'].value_counts()

Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64

In [4]:
df['MaritalStatus'].value_counts()

MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64

In [5]:
df['Gender'] = df['Gender'].replace('Fe Male', 'Female')
df['MaritalStatus'] = df['MaritalStatus'].replace('Singel', 'Unmarried')

In [6]:
df['Gender'].value_counts()

Gender
Male      2916
Female    1972
Name: count, dtype: int64

In [7]:
df.isnull().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [11]:
features_with_na = [features for features in df.columns if df[features].isnull().sum() >= 1]
for feature in features_with_na:
    print(feature, np.round(df[feature].isnull().mean()*100, 3), '% missing value')

Age 4.624 % missing value
TypeofContact 0.511 % missing value
DurationOfPitch 5.135 % missing value
NumberOfFollowups 0.921 % missing value
PreferredPropertyStar 0.532 % missing value
NumberOfTrips 2.864 % missing value
NumberOfChildrenVisiting 1.35 % missing value
MonthlyIncome 4.767 % missing value


In [12]:
df[features_with_na].select_dtypes(exclude='object').describe()

,Age,DurationOfPitch,NumberOfFollowups,PreferredPropertyStar,NumberOfTrips,NumberOfChildrenVisiting,MonthlyIncome
count,4662.000000,4637.000000,4843.000000,4862.000000,4748.000000,4822.000000,4655.000000
mean,37.622265,15.490835,3.708445,3.581037,3.236521,1.187267,23619.853491
std,9.316387,8.519643,1.002509,0.798009,1.849019,0.857861,5380.698361
min,18.000000,5.000000,1.000000,3.000000,1.000000,0.000000,1000.000000
25%,31.000000,9.000000,3.000000,3.000000,2.000000,1.000000,20346.000000
50%,36.000000,13.000000,4.000000,3.000000,3.000000,1.000000,22347.000000
75%,44.000000,20.000000,4.000000,4.000000,4.000000,2.000000,25571.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,3.000000,98678.000000


In [13]:
#Age
df.Age.fillna(df.Age.median(), inplace=True)

#TypeofContract
df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)

#DurationOfPitch
df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True)

#NumberOfFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)

#PreferredPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)

#NumberOfTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)

#NumberOfChildrenVisiting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)

#MonthlyIncome
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)

In [14]:
df.drop(['CustomerID'], axis=1, inplace=True)

In [15]:
df.head()

,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0


In [17]:
df['TotalVisiting'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting']
df.drop(columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'], axis=1, inplace=True)

In [18]:
## Get Numeric, Categorical, Discrete, Continous Feature

Numerical_feature = [feature for feature in df.columns if df[feature].dtype != 'O']
print('No. of numeric feature ', len(Numerical_feature))

Categorical_feature = [feature for feature in df.columns if df[feature].dtype == 'O']
print('No. of Categorical feature ', len(Categorical_feature))

discrete_feature = [feature for feature in Numerical_feature if len(df[feature].unique()) <= 25]
print('No. of Discrete feature ', len(discrete_feature))

continuous_feature = [feature for feature in Numerical_feature if feature not in discrete_feature]
print('No. of continuous feature ', len(continuous_feature))

No. of numeric feature  12
No. of Categorical feature  6
No. of Discrete feature  9
No. of continuous feature  3


In [19]:
X = df.drop(['ProdTaken'], axis=1)
y = df['ProdTaken']

In [20]:
X.shape

(4888, 17)

In [21]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)
X_train.shape

(3666, 17)

In [22]:
## create column transformer with 3 types of transformer

Categorical_feature = X.select_dtypes(include='object').columns
Numerical_feature = X.select_dtypes(exclude='object').columns

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer

Numerical_trans = StandardScaler()
oh_trans = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ('OneHotEncoder', oh_trans, Categorical_feature),
        ('StandardScaler', Numerical_trans, Numerical_feature)
    ]
)

In [23]:
preprocessor

,transformers,"[('OneHotEncoder', ...), ('StandardScaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,'first'
,sparse_output,True


In [24]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [25]:
pd.DataFrame(X_train)

,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,24,25,26
0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.721708,-0.653318,0.277912,1.777611,2.053422,1.575272,0.681958,-1.273702,-0.415942,-0.058810
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,-0.721708,-0.165525,-0.723883,-0.724971,-0.670111,-0.634811,1.409353,0.785113,-0.224146,-0.768009
2,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.721708,-0.287473,1.279708,0.526320,1.508716,-0.634811,0.681958,0.785113,-0.711215,-0.768009
3,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,1.454995,-0.531370,0.277912,1.777611,-0.670111,-0.634811,1.409353,0.785113,0.057482,-0.058810
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.721708,0.322268,-0.723883,-0.724971,-0.670111,1.575272,0.681958,0.785113,-1.139911,-0.058810
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3661,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.721708,-0.653318,1.279708,-0.724971,-0.670111,-0.634811,-1.500228,0.785113,-0.531928,0.650390
3662,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.454995,-0.897214,-0.723883,1.777611,-1.214818,-0.634811,1.409353,0.785113,1.528543,-0.058810
3663,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.454995,1.541750,0.277912,-0.724971,2.053422,-0.634811,-0.772833,0.785113,-0.356053,0.650390
3664,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.454995,1.785647,1.279708,-0.724971,-0.125404,-0.634811,-1.500228,0.785113,-0.248595,0.650390


In [27]:
## XG Boost Training

from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, precision_score, \
                            recall_score, roc_auc_score, roc_curve, f1_score

In [28]:
models = {
    'Logistic Regression' : LogisticRegression(),
    'GradientBoosting Classifier' : GradientBoostingClassifier(),
    'Ada Boost' : AdaBoostClassifier(),
    'RandomForest Classifier' : RandomForestClassifier(),
    'DecisionTree Classifier' : DecisionTreeClassifier(),
    'XG Boost' : XGBClassifier()
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    ##Make Prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## calculate score
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    train_precision = precision_score(y_train, y_train_pred)
    train_recall = recall_score(y_train, y_train_pred)
    train_roc_auc = roc_auc_score(y_train, y_train_pred)

    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_roc_auc = roc_auc_score(y_test, y_test_pred)
    

    print('----------Model Train Performance-----------')
    print('- Accuracy : {:.2f}'.format(train_accuracy))
    print('- F1-score : {:.2f}'.format(train_f1))
    print('- Precision : {:.2f}'.format(train_precision))
    print('- Recall : {:.2f}'.format(train_recall))
    print('- Roc_Auc : {:.2f}'.format(train_roc_auc))

    print('----------Model Test Performance-----------')
    print('- Accuracy : {:.2f}'.format(test_accuracy))
    print('- F1-score : {:.2f}'.format(test_f1))
    print('- Precision : {:.2f}'.format(test_precision))
    print('- Recall : {:.2f}'.format(test_recall))
    print('- Roc_Auc : {:.2f}'.format(test_roc_auc))

    print('='*25)
    print('\n')

----------Model Train Performance-----------
- Accuracy : 0.84
- F1-score : 0.82
- Precision : 0.70
- Recall : 0.31
- Roc_Auc : 0.64
----------Model Test Performance-----------
- Accuracy : 0.84
- F1-score : 0.82
- Precision : 0.65
- Recall : 0.32
- Roc_Auc : 0.64


----------Model Train Performance-----------
- Accuracy : 0.89
- F1-score : 0.88
- Precision : 0.88
- Recall : 0.49
- Roc_Auc : 0.74
----------Model Test Performance-----------
- Accuracy : 0.87
- F1-score : 0.86
- Precision : 0.79
- Recall : 0.42
- Roc_Auc : 0.70


----------Model Train Performance-----------
- Accuracy : 0.85
- F1-score : 0.82
- Precision : 0.77
- Recall : 0.30
- Roc_Auc : 0.64
----------Model Test Performance-----------
- Accuracy : 0.85
- F1-score : 0.82
- Precision : 0.72
- Recall : 0.27
- Roc_Auc : 0.62


----------Model Train Performance-----------
- Accuracy : 1.00
- F1-score : 1.00
- Precision : 1.00
- Recall : 1.00
- Roc_Auc : 1.00
----------Model Test Performance-----------
- Accuracy : 0.92
- F1

In [29]:
rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "auto", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}
xg_params = {"learning_rate": [0.1, 0.01],
              "max_depth": [5, 8, 12, 20, 30],
              "n_estimators": [100, 200, 300],
              "colsample_bytree": [0.5, 0.8, 1, 0.3, 0.4]}

In [31]:
randomcv_models = [
    ('RF', RandomForestClassifier(), rf_params),
    ('XG', XGBClassifier(), xg_params)
]
    

In [32]:

from sklearn.model_selection import RandomizedSearchCV
model_param = {}

for name,model,params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                   param_distributions=params,
                                   n_iter=100,
                                   cv=3,
                                   verbose=2,
                                   n_jobs=-1)
    
    random.fit(X_train,y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f'----------Best Params {model_name}-----------')
    print(model_param[model_name])

Fitting 3 folds for each of 100 candidates, totalling 300 fits
Fitting 3 folds for each of 100 candidates, totalling 300 fits
----------Best Params RF-----------
{'n_estimators': 100, 'min_samples_split': 2, 'max_features': 7, 'max_depth': None}
----------Best Params XG-----------
{'n_estimators': 300, 'max_depth': 30, 'learning_rate': 0.1, 'colsample_bytree': 0.5}


In [36]:
models = {
    'RandomForest Classifier' : RandomForestClassifier(n_estimators=100, min_samples_split=2,
                                                        max_features=7, max_depth=None, n_jobs=-1),
    'XG Boost' : XGBClassifier(n_estimators=300, max_depth= 30, learning_rate= 0.1, colsample_bytree= 0.5)
}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    ##Make Prediction
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    ## calculate score
    train_accuracy = accuracy_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    train_precision = precision_score(y_train, y_train_pred)
    train_recall = recall_score(y_train, y_train_pred)
    train_roc_auc = roc_auc_score(y_train, y_train_pred)

    test_accuracy = accuracy_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_roc_auc = roc_auc_score(y_test, y_test_pred)
    

    print('----------Model Train Performance-----------')
    print('- Accuracy : {:.2f}'.format(train_accuracy))
    print('- F1-score : {:.2f}'.format(train_f1))
    print('- Precision : {:.2f}'.format(train_precision))
    print('- Recall : {:.2f}'.format(train_recall))
    print('- Roc_Auc : {:.2f}'.format(train_roc_auc))

    print('----------Model Test Performance-----------')
    print('- Accuracy : {:.2f}'.format(test_accuracy))
    print('- F1-score : {:.2f}'.format(test_f1))
    print('- Precision : {:.2f}'.format(test_precision))
    print('- Recall : {:.2f}'.format(test_recall))
    print('- Roc_Auc : {:.2f}'.format(test_roc_auc))

    print('='*25)
    print('\n')

----------Model Train Performance-----------
- Accuracy : 1.00
- F1-score : 1.00
- Precision : 1.00
- Recall : 1.00
- Roc_Auc : 1.00
----------Model Test Performance-----------
- Accuracy : 0.92
- F1-score : 0.92
- Precision : 0.94
- Recall : 0.63
- Roc_Auc : 0.81


----------Model Train Performance-----------
- Accuracy : 1.00
- F1-score : 1.00
- Precision : 1.00
- Recall : 1.00
- Roc_Auc : 1.00
----------Model Test Performance-----------
- Accuracy : 0.94
- F1-score : 0.94
- Precision : 0.94
- Recall : 0.71
- Roc_Auc : 0.85


